# B05 — Điều tra vì sao albopictus thất bại khi chuyển nguồn thu

**Câu hỏi:** model đã học được albopictus ở train nhưng thất bại ở validation vì khác nguồn ghi, tín hiệu/nhiễu, giới tính/nhãn, hay lỗi đánh giá? Notebook kiểm tra bằng chứng; không mặc định nguyên nhân.

- Đọc kết quả **full** của B và giữ nguyên A_v1; chỉ nghe/đo audio train/validation.
- Phân biệt **window 300 ms** và **clip WAV BioDCASE**, không nhầm clip với phiên thu gốc.
- Một nguồn có nhiều clip vẫn có thể là một bằng chứng nguồn duy nhất. Không tính khoảng tin cậy bằng cách giả định các window overlap độc lập.
- Full: một window ở giữa mỗi clip đủ dài để khảo sát phổ; không gọi đây là phân tích mọi window. Smoke chỉ lấy ít clip theo loài/split/domain và không dùng để kết luận phân bố.
- Output: bảng CSV, hình, audio minh họa và `report.html` trong `results/B/domain_diagnostics/<mode>/<run_id>/`. Khi chạy Colab có snapshot trên Drive.

## 0. Thiết lập và chuẩn bị môi trường
Sửa tham số trực tiếp ở cell dưới. Colab mount shortcut `Edge_AI_Mosquito`, clone/pull repo và chỉ phục hồi data/A từ bundle đã kiểm checksum. Local dùng dữ liệu trong project hoặc đường dẫn raw đã ghi trong manifest. Không cần cài thêm thư viện ngoài môi trường B.

In [1]:
from pathlib import Path, PureWindowsPath, PurePosixPath
import os, sys, subprocess, shutil, csv, hashlib, io, json, re, time, zipfile
import importlib.util, importlib.metadata
EXECUTION_ENV='auto'  # auto / local / colab
DRIVE_FOLDER=Path('/content/drive/MyDrive/Edge_AI_Mosquito')
COLAB_PROJECT_DIR=Path('/content/Edge_AI_Mosquito')
REPO_URL='https://github.com/ancaranovik/Edge_AI_Mosquito.git';REPO_BRANCH='main'
BUNDLE_SHA256='29a099018450a1dc1e54201c58d441a72d53ec0c8ace10b8921b40c4c34c6278'
SAVE_TO_DRIVE=True
CPU_THREADS=4
NOTEBOOK_NAME='05_B_domain_diagnostics.ipynb'
STAGE='domain_diagnostics'
RUN_MODE='full'
# Full ở 05 = phân tích toàn bộ train/validation, không huấn luyện.
SEED=42
SMOKE_CLIPS_PER_GROUP=4
BASELINE_RUN_ID='b67e5bff23cad061fc1a59accc6ec3661e137551efa84979292ea86a4a205112'
BASELINE_REPORT=None  # None tự tìm; hoặc Path tới report.zip / folder chứa tables của full B.

os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG',':4096:8')
assert EXECUTION_ENV in ('auto','local','colab') and RUN_MODE in ('smoke','full')
try:has_colab=importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:has_colab=False
IS_COLAB=EXECUTION_ENV=='colab' or (EXECUTION_ENV=='auto' and has_colab and Path('/content').is_dir())
print('Môi trường:', 'Colab' if IS_COLAB else 'local','| Mode:',RUN_MODE)

Môi trường: Colab | Mode: full


In [2]:
def bootstrap_sha(path):
    with Path(path).open('rb') as f: return hashlib.file_digest(f,'sha256').hexdigest()

def archive_relative(name):
    rel=PurePosixPath(name)
    if not name or rel.is_absolute() or '..' in rel.parts or chr(92) in name or ':' in name:
        raise ValueError('Đường dẫn archive không hợp lệ: '+name)
    return rel

def extract_dataset_bundle(archive,project_root):
    project_root=Path(project_root).resolve()
    with zipfile.ZipFile(archive) as z:
        names=z.namelist()
        assert len(names)==len(set(names)), 'Archive trùng tên file'
        for name in names: archive_relative(name)
        rows=list(csv.DictReader(io.StringIO(z.read('bundle_inventory.csv').decode('utf-8-sig'))))
        selected=[r for r in rows if r['archive_path'].startswith(('data/','results/A/'))]
        assert len({r['archive_path'] for r in selected})==len(selected)
        assert {n for n in names if n.startswith(('data/','results/A/'))}=={r['archive_path'] for r in selected}
        assert sum(r['type']=='audio' for r in selected)==2905
        needed=sum(int(r['bytes']) for r in selected if not project_root.joinpath(*archive_relative(r['archive_path']).parts).exists())
        if shutil.disk_usage(project_root).free < needed+100_000_000:
            raise OSError('Ổ tạm không đủ chỗ giải nén gói A_v1')
        reused=written=0
        for i,row in enumerate(selected,1):
            rel=archive_relative(row['archive_path']);target=project_root.joinpath(*rel.parts)
            assert target.resolve().is_relative_to(project_root)
            if target.is_file() and target.stat().st_size==int(row['bytes']) and bootstrap_sha(target)==row['sha256']:
                reused+=1;continue
            target.parent.mkdir(parents=True,exist_ok=True)
            tmp=target.with_name(target.name+'.extracting')
            h=hashlib.sha256();size=0
            with z.open(row['archive_path']) as src,tmp.open('wb') as dst:
                while chunk:=src.read(1024*1024): dst.write(chunk);h.update(chunk);size+=len(chunk)
            if h.hexdigest()!=row['sha256'] or size!=int(row['bytes']):
                raise ValueError('Hash giải nén sai: '+str(rel))
            os.replace(tmp,target);written+=1
            if i%500==0: print('Giải nén/kiểm tra:',i,'/',len(selected),flush=True)
    return dict(bundle_files=len(selected),audio_files=2905,files_written=written,files_reused=reused)

bootstrap_info=dict(environment='colab' if IS_COLAB else 'local',bundle_sha256='not_used')
if IS_COLAB:
    if not (DRIVE_FOLDER/'A_v1_bundle.zip').is_file():
        from google.colab import drive
        drive.mount('/content/drive')
    source_bundle=DRIVE_FOLDER/'A_v1_bundle.zip'
    if not source_bundle.is_file():
        raise FileNotFoundError(f'Không thấy {source_bundle}. Kiểm tra tên shortcut trong My Drive A.')
    ROOT=COLAB_PROJECT_DIR.resolve()
    if not (ROOT/'.git').exists():
        if ROOT.exists() and any(ROOT.iterdir()):
            raise RuntimeError('Folder Colab đã có file nhưng chưa có Git; chọn COLAB_PROJECT_DIR khác.')
        subprocess.run(['git','clone','--branch',REPO_BRANCH,REPO_URL,str(ROOT)],check=True)
    else:
        remote=subprocess.check_output(['git','-C',str(ROOT),'remote','get-url','origin'],text=True).strip()
        assert remote.rstrip('/').removesuffix('.git')==REPO_URL.rstrip('/').removesuffix('.git')
        dirty=subprocess.check_output(['git','-C',str(ROOT),'status','--porcelain','--untracked-files=no'],text=True)
        if dirty.strip(): raise RuntimeError('Code trên runtime đã sửa. Giữ bản sửa trước khi pull; không tự reset.')
        branch=subprocess.check_output(['git','-C',str(ROOT),'branch','--show-current'],text=True).strip()
        assert branch==REPO_BRANCH,'Checkout branch khác; không tự đổi hoặc ghi đè.'
        subprocess.run(['git','-C',str(ROOT),'pull','--ff-only','origin',REPO_BRANCH],check=True)
    staging=ROOT/'data/.bundle';staging.mkdir(parents=True,exist_ok=True)
    local_bundle=staging/'A_v1_bundle.zip'
    if not local_bundle.is_file() or bootstrap_sha(local_bundle)!=BUNDLE_SHA256:
        if shutil.disk_usage(ROOT).free < source_bundle.stat().st_size+100_000_000:
            raise OSError('Ổ tạm không đủ chỗ copy archive')
        tmp=local_bundle.with_suffix('.zip.copying')
        shutil.copyfile(source_bundle,tmp)
        assert bootstrap_sha(tmp)==BUNDLE_SHA256,'Archive trên Drive chưa khớp gói A_v1 đã chốt'
        os.replace(tmp,local_bundle)
    bootstrap_info.update(extract_dataset_bundle(local_bundle,ROOT),bundle_sha256=BUNDLE_SHA256)
    os.chdir(ROOT)
else:
    ROOT=next((p for p in [Path.cwd(),*Path.cwd().parents] if (p/'notebooks').is_dir() and (p/'README.md').is_file()),None)
    if ROOT is None: raise FileNotFoundError('Mở notebook từ folder repo local.')
    ROOT=ROOT.resolve()
try:
    bootstrap_info['git_commit']=subprocess.check_output(['git','-C',str(ROOT),'rev-parse','HEAD'],text=True,stderr=subprocess.DEVNULL).strip()
except subprocess.CalledProcessError:
    bootstrap_info['git_commit']='not_available'
print('Project trên kernel:',ROOT)
print(bootstrap_info)

Mounted at /content/drive
Giải nén/kiểm tra: 500 / 3032
Giải nén/kiểm tra: 1000 / 3032
Giải nén/kiểm tra: 1500 / 3032
Giải nén/kiểm tra: 2000 / 3032
Giải nén/kiểm tra: 2500 / 3032
Giải nén/kiểm tra: 3000 / 3032
Project trên kernel: /content/Edge_AI_Mosquito
{'environment': 'colab', 'bundle_sha256': '29a099018450a1dc1e54201c58d441a72d53ec0c8ace10b8921b40c4c34c6278', 'bundle_files': 3032, 'audio_files': 2905, 'files_written': 3032, 'files_reused': 0, 'git_commit': '39c0d028def1e0c72ced4329d7d13cbcf6818108'}


In [3]:
packages={'numpy':'numpy','pandas':'pandas','scipy':'scipy','soundfile':'soundfile','matplotlib':'matplotlib','seaborn':'seaborn','scikit-learn':'sklearn','nbformat':'nbformat','torch':'torch'}
missing=[p for p,m in packages.items() if importlib.util.find_spec(m) is None]
if missing:
    if not IS_COLAB:raise ImportError('Thiếu thư viện: '+', '.join(missing)+'. Dùng .venv/requirements.txt.')
    if 'torch' in missing:raise ImportError('Chọn runtime Colab có PyTorch; không cài bản CPU từ requirements vào Colab.')
    subprocess.run([sys.executable,'-m','pip','install','--quiet',*missing],check=True)

## 1. Hàm dùng chung và xác minh A_v1
Các cell thuật toán phải khớp source trên kernel (quan trọng khi dùng VS Code nối Colab). Giữ nhãn, split và biên window của A. Loader từ chối audio test trước khi truy cập file; resample/canonical hash phải khớp A.

In [4]:
from math import gcd
from functools import lru_cache
from datetime import datetime
import copy, html, platform, random
import nbformat
import numpy as np
import pandas as pd
import scipy
from scipy import signal
import soundfile as sf
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.metrics import f1_score, confusion_matrix, precision_recall_fscore_support
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from IPython.display import display, Markdown, Audio
LABEL_NAMES=['Aedes aegypti','Aedes albopictus','Anopheles dirus','Anopheles minimus']
LABELS=np.arange(4)
SAMPLE_RATE=8000; WINDOW_SAMPLES=2400; STFT_N=256; STFT_HOP=80; MEL_BINS=40
PSD_FLOOR=1e-12
versions=dict(python=platform.python_version(),numpy=np.__version__,pandas=pd.__version__,
              scipy=scipy.__version__,soundfile=sf.__version__,sklearn=sklearn.__version__,torch=str(torch.__version__))
sns.set_theme(style='whitegrid');plt.rcParams.update({'figure.dpi':110,'axes.spines.top':False,'axes.spines.right':False})
def sha(path):
    with Path(path).open('rb') as f:return hashlib.file_digest(f,'sha256').hexdigest()
def digest(value):return hashlib.sha256(json.dumps(value,sort_keys=True,ensure_ascii=False,default=str).encode()).hexdigest()
def seed_all(seed):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed)
    if torch.cuda.is_available():torch.cuda.manual_seed_all(seed)
NOTEBOOK_PATH=ROOT/'notebooks'/NOTEBOOK_NAME
notebook_source=nbformat.read(NOTEBOOK_PATH,as_version=4)
core_cells={c.metadata.tags[0]:c.source for c in notebook_source.cells if c.cell_type=='code' and c.metadata.get('tags') and c.metadata.tags[0] not in ('settings','bootstrap','dependencies')}
CODE_SHA=digest(core_cells); verified_tags=set()
def verify_cell(tag):
    from IPython import get_ipython
    shell=get_ipython()
    if shell is not None and shell.history_manager.input_hist_raw:
        actual=shell.history_manager.input_hist_raw[-1].replace('\r\n','\n').strip()
        if actual!=core_cells[tag].replace('\r\n','\n').strip():
            raise RuntimeError('Cell khác source trên kernel: '+tag+'. Save/push/pull notebook trước khi chạy; tham số được sửa ở cell đầu.')
    verified_tags.add(tag)
def atomic_bytes(path,value):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True);tmp=path.with_name(path.name+'.tmp')
    tmp.write_bytes(value)
    for attempt in range(6):
        try:os.replace(tmp,path);return
        except PermissionError:
            if attempt==5:raise
            time.sleep(.05*2**attempt)
def save_table(frame,name):
    atomic_bytes(TABLES/(name+'.csv'),frame.to_csv(index=False).encode('utf-8-sig'));return frame
def save_figure(name):
    plt.tight_layout();plt.savefig(FIGURES/(name+'.png'),dpi=150,bbox_inches='tight');plt.show();plt.close()
checks=[]
def check(name,condition):
    checks.append(dict(check=name,passed=bool(condition)))
    if not condition:raise AssertionError(name)
CLIP_PATH=ROOT/'data/manifests/biodcase_four_species_clips.csv'
WINDOW_PATH=ROOT/'data/manifests/biodcase_four_species_windows.csv'
FREEZE_PATH=ROOT/'results/A/preparation/tables/dataset_freeze.csv'
PROTECTED={p:sha(p) for p in [CLIP_PATH,WINDOW_PATH,FREEZE_PATH,ROOT/'notebooks/03_A_data_preparation.ipynb',ROOT/'notebooks/04_B_baseline.ipynb']}
def init_run(settings):
    global RUN_ID,RUN_OUT,TABLES,FIGURES,CACHE,REMOTE
    settings=dict(settings,source_sha=CODE_SHA,versions=versions,freeze_sha=sha(FREEZE_PATH),
                  manifests=[sha(CLIP_PATH),sha(WINDOW_PATH)])
    RUN_ID=digest(settings);RUN_OUT=ROOT/'results/B'/STAGE/RUN_MODE/RUN_ID[:16]
    TABLES=RUN_OUT/'tables';FIGURES=RUN_OUT/'figures';CACHE=RUN_OUT/'cache'
    for folder in [TABLES,FIGURES,CACHE]:folder.mkdir(parents=True,exist_ok=True)
    REMOTE=DRIVE_FOLDER/'runs/B'/STAGE/RUN_MODE/RUN_ID[:16] if IS_COLAB and SAVE_TO_DRIVE else None
    save_table(pd.DataFrame([dict(item=k,value=json.dumps(v,ensure_ascii=False,default=str)) for k,v in settings.items()]+[dict(item='run_id',value=RUN_ID)]),'run_settings')
    atomic_bytes(RUN_OUT/'source_notebook.ipynb',nbformat.writes(notebook_source).encode('utf-8'))
    if REMOTE is not None:
        REMOTE.mkdir(parents=True,exist_ok=True);probe=REMOTE/'write_probe.tmp'
        probe.write_bytes(b'diagnostic write check');assert probe.read_bytes()==b'diagnostic write check';probe.unlink()
    print('Run:',RUN_ID[:16],'|',RUN_MODE,'| Output:',RUN_OUT)
    if REMOTE is not None:print('Drive:',REMOTE)
verify_cell('helpers')

FileNotFoundError: [Errno 2] No such file or directory: '/content/Edge_AI_Mosquito/notebooks/05_B_domain_diagnostics.ipynb'

In [ ]:
verify_cell('data')
def frozen_path(value):
    rel=PureWindowsPath(str(value));assert not rel.is_absolute() and not rel.drive and '..' not in rel.parts
    return ROOT.joinpath(*rel.parts)
freeze=pd.read_csv(FREEZE_PATH,keep_default_na=False).set_index('item').value.to_dict()
assert freeze['protocol_version']=='A_v1'
a_source=nbformat.read(ROOT/'notebooks/03_A_data_preparation.ipynb',as_version=4)
assert hashlib.sha256('\n'.join(c.source for c in a_source.cells).encode()).hexdigest()==freeze['notebook_source_sha256']
for key,value in freeze.items():
    if key.endswith('_path'):assert sha(frozen_path(value))==freeze[key[:-5]+'_sha256'],key
clips=pd.read_csv(CLIP_PATH,keep_default_na=False);windows=pd.read_csv(WINDOW_PATH)
kept=clips[clips.status.eq('keep')].copy()
assert clips.clip_id.is_unique and windows.window_id.is_unique
assert kept.groupby('leakage_group').split.nunique().le(1).all()
assert kept[['label_id','species']].drop_duplicates().set_index('label_id').species.to_dict()==dict(enumerate(LABEL_NAMES))
assert windows.end_sample.sub(windows.start_sample).eq(WINDOW_SAMPLES).all() and windows.sample_rate.eq(SAMPLE_RATE).all()
joined=windows.merge(kept[['clip_id','label_id','species','split','processed_frames']],on='clip_id',suffixes=('','_clip'),validate='many_to_one')
for col in ['label_id','species','split']:assert joined[col].eq(joined[col+'_clip']).all()
assert joined.end_sample.le(joined.processed_frames).all()
# Metadata test có trong manifest để kiểm tra split; audio lookup chỉ chứa train/validation.
allowed_windows=windows[windows.split.isin(['train','validation'])].copy()
test_ids=set(kept[kept.split.eq('test')].clip_id)
allowed_ids=set(allowed_windows.clip_id)
clip_lookup=kept[kept.clip_id.isin(allowed_ids)].set_index('clip_id').copy()
assert not allowed_ids&test_ids and set(clip_lookup.index)==allowed_ids
assert clips.path.map(lambda p:PureWindowsPath(p).name).is_unique
accessed=set()
def audio_path(row):
    candidate=ROOT/'data/audio/biodcase_A_v1'/PureWindowsPath(row.path).name
    if candidate.is_file():return candidate
    if not IS_COLAB and Path(row.path).is_file():return Path(row.path)
    raise FileNotFoundError('Không thấy WAV A_v1: '+str(candidate))
@lru_cache(maxsize=2)
def read_clip(clip_id):
    if clip_id not in allowed_ids or clip_id in test_ids:raise ValueError('Không được đọc audio ngoài train/validation: '+str(clip_id))
    row=clip_lookup.loc[clip_id];p=audio_path(row)
    assert sha(p)==row.sha256,'Raw WAV thay đổi: '+clip_id
    x,sr=sf.read(p,dtype='float64',always_2d=True)
    assert len(x)>0 and np.isfinite(x).all();x=x.mean(axis=1)
    if sr!=SAMPLE_RATE:
        factor=gcd(int(sr),SAMPLE_RATE);x=signal.resample_poly(x,SAMPLE_RATE//factor,int(sr)//factor)
    x=np.asarray(x,dtype='<f4')
    actual=hashlib.sha256(f'{SAMPLE_RATE}|1|{len(x)}|'.encode()+x.tobytes()).hexdigest()
    assert actual==row.canonical_sha256 and len(x)==int(row.processed_frames),'Canonical khác A: '+clip_id
    accessed.add(clip_id);return x
HANN=signal.windows.hann(STFT_N,sym=False);FREQUENCIES=np.fft.rfftfreq(STFT_N,1/SAMPLE_RATE)
mel_edges=700*(10**(np.linspace(0,2595*np.log10(1+4000/700),MEL_BINS+2)/2595)-1)
mel_filter=np.maximum(0,np.minimum((FREQUENCIES[None,:]-mel_edges[:-2,None])/(mel_edges[1:-1,None]-mel_edges[:-2,None]),
    (mel_edges[2:,None]-FREQUENCIES[None,:])/(mel_edges[2:,None]-mel_edges[1:-1,None])))
mel_filter/=mel_filter.sum(axis=1,keepdims=True)
def logmel(x):
    frames=np.lib.stride_tricks.sliding_window_view(np.asarray(x,dtype=np.float64),STFT_N)[::STFT_HOP]
    power=np.abs(np.fft.rfft(frames*HANN,axis=-1))**2/(SAMPLE_RATE*np.sum(HANN**2));power[:,1:-1]*=2
    out=10*np.log10(np.maximum(mel_filter@power.T,PSD_FLOOR))
    assert out.shape==(40,27) and np.isfinite(out).all();return out.astype('<f4')
def aggregate_clips(frame,p):
    p=np.asarray(p,dtype=np.float64)
    assert p.shape==(len(frame),4) and np.isfinite(p).all() and np.allclose(p.sum(1),1,atol=1e-5)
    tmp=frame[['clip_id','label_id','species','domain','leakage_group']].copy()
    for i in LABELS:tmp[f'p{i}']=p[:,i]
    assert tmp.groupby('clip_id').label_id.nunique().eq(1).all()
    c=tmp.groupby('clip_id',sort=True).agg(label_id=('label_id','first'),species=('species','first'),domain=('domain','first'),
        leakage_group=('leakage_group','first'),windows=('label_id','size'),**{f'p{i}':(f'p{i}','mean') for i in LABELS}).reset_index()
    c['prediction']=c[[f'p{i}' for i in LABELS]].to_numpy().argmax(1);return c
check('Freeze A_v1 và nhãn cố định',freeze['protocol_version']=='A_v1')
try:read_clip(sorted(test_ids)[0]);blocked=False
except ValueError:blocked=True
check('Từ chối audio test trước I/O',blocked and not accessed)
print('Train/validation:',len(allowed_windows),'window;',len(allowed_ids),'clip đủ dài. Không đọc audio test.')

## 2. Đọc đúng kết quả full, kiểm tra xác suất và tính lại F1
Tự tìm kết quả full đúng run ID trên ổ local hoặc hai slot snapshot Drive của baseline. Kiểm inventory/hash trước khi đọc archive. Không nhận kết quả smoke và không tự chạy lại B khi thiếu report. Có thể khai báo `BASELINE_REPORT` ở cell đầu.

Tính lại `argmax`, nhãn, mean-probability theo clip, F1/recall và thống kê albopictus. Cùng nhãn dự đoán giữa hai model không có nghĩa chúng dùng chung xác suất.

In [ ]:
verify_cell('baseline_audit')
MODELS=['spectral_rf','small_cnn','ds_cnn','tc_resnet8','cnn_lstm','mosquitosong_style_1d','bc_resnet1','microdsc_style']
def validate_archive(path,expected_sha=None):
    if expected_sha is not None:assert sha(path)==expected_sha,'Archive checksum khác receipt'
    with zipfile.ZipFile(path) as z:
        names=z.namelist();assert len(names)==len(set(names))
        inv=pd.read_csv(io.BytesIO(z.read('snapshot_inventory.csv')))
        assert set(names)==set(inv.path)|{'snapshot_inventory.csv'}
        for r in inv.itertuples():
            archive_relative(r.path)
            with z.open(r.path) as f:assert hashlib.file_digest(f,'sha256').hexdigest()==r.sha256
            assert z.getinfo(r.path).file_size==r.bytes
    return Path(path)
def locate_baseline():
    if BASELINE_REPORT is not None:return Path(BASELINE_REPORT)
    local=ROOT/'results/B/baseline_v1/full/io_v2'
    if (local/'tables/run_settings.csv').is_file():return local
    archive=ROOT/'results/B/baseline_full_reference'/BASELINE_RUN_ID[:16]/'report.zip'
    if archive.is_file():return archive
    candidates=[]
    folder=DRIVE_FOLDER/'runs/B/baseline_v1/full'/BASELINE_RUN_ID[:16]/'report'
    for slot in range(2):
        receipt=folder/f'slot{slot}.csv';p=folder/f'slot{slot}.zip'
        if not receipt.is_file() or not p.is_file():continue
        try:
            rows=pd.read_csv(receipt);assert len(rows)==1;r=rows.iloc[0]
            assert r.run_id==BASELINE_RUN_ID and r['key']=='report' and p.stat().st_size==r['bytes']
            validate_archive(p,r.sha256);candidates.append((int(r.generation),p))
        except (AssertionError,OSError,ValueError,KeyError,zipfile.BadZipFile) as e:print('Bỏ slot không hợp lệ:',slot,type(e).__name__)
    if candidates:return max(candidates,key=lambda x:x[0])[1]
    raise FileNotFoundError('Không thấy report FULL đúng run ID. Chạy report cuối 04 hoặc khai báo BASELINE_REPORT tới snapshot đã tải; không dùng smoke thay thế.')
baseline_path=locate_baseline()
if baseline_path.is_file():
    validate_archive(baseline_path)
    # Copy nhỏ về ổ tạm/local để không đọc ZIP lặp qua mount Drive.
    target=ROOT/'results/B/baseline_full_reference'/BASELINE_RUN_ID[:16]/'report.zip'
    target.parent.mkdir(parents=True,exist_ok=True)
    if baseline_path.resolve()!=target.resolve():
        payload=baseline_path.read_bytes();atomic_bytes(target,payload);assert sha(target)==hashlib.sha256(payload).hexdigest()
    baseline_path=target
    def baseline_bytes(rel):
        archive_relative(rel)
        with zipfile.ZipFile(baseline_path) as z:return z.read(rel)
else:
    def baseline_bytes(rel):
        archive_relative(rel);return baseline_path.joinpath(*PurePosixPath(rel).parts).read_bytes()
def baseline_csv(name):return pd.read_csv(io.BytesIO(baseline_bytes('tables/'+name+'.csv')))
def decode_setting(value):
    try:return json.loads(value)
    except (ValueError,TypeError):return value
settings_b={r.item:decode_setting(r.value) for r in baseline_csv('run_settings').itertuples()}
assert settings_b['run_id']==BASELINE_RUN_ID and settings_b['mode']=='full'
assert settings_b['A_freeze_sha256']==sha(FREEZE_PATH)
metrics_b=baseline_csv('metrics');perclass_b=baseline_csv('per_class_metrics')
assert metrics_b['mode'].eq('full').all() and set(metrics_b.model)==set(MODELS)|{'majority'}
# Full B phải dùng đúng toàn bộ validation đang khóa.
val=allowed_windows[allowed_windows.split.eq('validation')].sort_values(['clip_id','start_sample']).reset_index(drop=True)
expected_windows=val.set_index('window_id');expected_clips=clip_lookup.loc[val.clip_id.unique()]
prob_cols=['p0','p1','p2','p3'];predictions_b={};window_predictions_b={};audit=[];alb_rows=[];agreement=[]
for model in MODELS:
    w=baseline_csv(model+'_validation_windows');c=baseline_csv(model+'_validation_clips').sort_values('clip_id').reset_index(drop=True)
    assert w.window_id.is_unique and c.clip_id.is_unique
    assert set(w.window_id)==set(val.window_id) and set(c.clip_id)==set(val.clip_id)
    assert w['mode'].eq('full').all() and c['mode'].eq('full').all() and w.model.eq(model).all() and c.model.eq(model).all()
    w=w.set_index('window_id').loc[val.window_id].reset_index()
    for col in ['clip_id','label_id','domain']:assert np.array_equal(w[col].to_numpy(),val[col].to_numpy())
    assert np.array_equal(c.label_id,clip_lookup.loc[c.clip_id,'label_id'])
    assert np.array_equal(c.domain,clip_lookup.loc[c.clip_id,'domain'])
    p=w[prob_cols].to_numpy();cp=c[prob_cols].to_numpy()
    assert np.isfinite(p).all() and np.isfinite(cp).all() and (p>=-1e-8).all() and (p<=1+1e-8).all()
    assert np.allclose(p.sum(1),1,atol=1e-5) and np.allclose(cp.sum(1),1,atol=1e-5)
    assert np.array_equal(p.argmax(1),w.prediction) and np.array_equal(cp.argmax(1),c.prediction)
    aggregated=aggregate_clips(val,p)
    assert np.allclose(cp,aggregated[prob_cols],atol=1e-6) and np.array_equal(c.windows,aggregated.windows)
    for level,frame in [('window',w),('clip',c)]:
        score=f1_score(frame.label_id,frame.prediction,labels=LABELS,average='macro',zero_division=0)
        reported=metrics_b.query('model==@model and split=="validation" and level==@level').macro_f1.item()
        assert np.isclose(score,reported,atol=1e-10)
        audit.append(dict(model=model,level=level,examples=len(frame),macro_f1=score,metric_matches=True))
    alb=c[c.label_id.eq(1)];aw=w[w.label_id.eq(1)]
    tr=perclass_b.query('model==@model and split=="train" and level=="clip" and species=="Aedes albopictus"')
    alb_rows.append(dict(model=model,train_clip_recall=tr.recall.item(),validation_clip_recall=float(alb.prediction.eq(1).mean()),
        validation_window_recall=float(aw.prediction.eq(1).mean()),validation_clips=len(alb),validation_windows=len(aw),
        predicted_minimus_clips=int(alb.prediction.eq(3).sum()),mean_p_albopictus=alb.p1.mean(),max_p_albopictus=alb.p1.max(),
        mean_p_minimus=alb.p3.mean(),minimus_score_over_95_clips=int(alb.p3.gt(.95).sum())))
    predictions_b[model]=c;window_predictions_b[model]=w
for model,c in predictions_b.items():
    ref=predictions_b['ds_cnn'];agreement.append(dict(model=model,same_clip_labels=int(c.prediction.eq(ref.prediction).sum()),
        total=len(c),max_abs_probability_difference=float(np.abs(c[prob_cols].to_numpy()-ref[prob_cols].to_numpy()).max())))
input_hashes={name:hashlib.sha256(baseline_bytes('tables/'+name+'.csv')).hexdigest() for name in ['run_settings','metrics','per_class_metrics']+[m+'_validation_clips' for m in MODELS]+[m+'_validation_windows' for m in MODELS]}
input_hashes['models/ds_cnn/best.pt']=hashlib.sha256(baseline_bytes('models/ds_cnn/best.pt')).hexdigest()
init_run(dict(stage=STAGE,mode=RUN_MODE,seed=SEED,smoke_clips_per_group=SMOKE_CLIPS_PER_GROUP,baseline_run_id=BASELINE_RUN_ID,input_hashes=input_hashes))
check('Tám model full và metric tính lại đúng',len(audit)==16 and all(r['metric_matches'] for r in audit))
save_table(pd.DataFrame(audit),'baseline_metric_audit');alb_summary=save_table(pd.DataFrame(alb_rows),'albopictus_failure_summary')
save_table(pd.DataFrame(agreement),'prediction_agreement');save_table(pd.DataFrame([dict(file=k,sha256=v) for k,v in input_hashes.items()]),'baseline_inputs')
display(alb_summary);display(pd.DataFrame(agreement))
print('Baseline đã kiểm:',baseline_path)

In [ ]:
verify_cell('persistence')
persistence_events=[]
def read_snapshot(key):
    if REMOTE is None:return None
    valid=[];seen=False
    for slot in range(2):
        folder=REMOTE/key;receipt=folder/f'slot{slot}.csv';archive=folder/f'slot{slot}.zip'
        if not receipt.exists() and not archive.exists():continue
        seen=True
        try:
            rec=pd.read_csv(receipt,keep_default_na=False);assert len(rec)==1;r=rec.iloc[0]
            assert r.run_id==RUN_ID and r['key']==key
            local=CACHE/f'{key}_slot{slot}.zip';shutil.copyfile(archive,local)
            assert local.stat().st_size==int(r['bytes']) and sha(local)==r.sha256
            with zipfile.ZipFile(local) as z:
                names=z.namelist();assert len(names)==len(set(names))
                inventory=pd.read_csv(io.BytesIO(z.read('snapshot_inventory.csv')))
                assert set(names)==set(inventory.path)|{'snapshot_inventory.csv'}
                for row in inventory.itertuples():
                    rel=archive_relative(row.path);assert rel.parts[0] in ('tables','models','figures') or row.path in ('report.html','source_notebook.ipynb')
                    payload=z.read(row.path);assert len(payload)==row.bytes and hashlib.sha256(payload).hexdigest()==row.sha256
            valid.append((int(r.generation),local))
        except (OSError,AssertionError,ValueError,KeyError,zipfile.BadZipFile) as e:
            persistence_events.append(dict(key=key,event='invalid_slot',detail=f'{slot}: {type(e).__name__}'))
    if seen and not valid:raise RuntimeError('Hai snapshot đều hỏng: '+key+'. Không tự ghi đè hoặc train lại.')
    return max(valid,key=lambda v:v[0]) if valid else None
def save_snapshot(key,files,progress):
    if REMOTE is None:return
    latest=read_snapshot(key);generation=(latest[0]+1) if latest else 1;slot=generation%2
    archive=CACHE/f'{key}_writing.zip';inventory=[]
    with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=1) as z:
        for path in sorted(set(map(Path,files))):
            rel=path.relative_to(RUN_OUT).as_posix();archive_relative(rel);payload=path.read_bytes()
            inventory.append(dict(path=rel,bytes=len(payload),sha256=hashlib.sha256(payload).hexdigest()));z.writestr(rel,payload)
        z.writestr('snapshot_inventory.csv',pd.DataFrame(inventory).to_csv(index=False))
    folder=REMOTE/key;folder.mkdir(parents=True,exist_ok=True);remote=folder/f'slot{slot}.zip'
    shutil.copyfile(archive,remote);assert sha(remote)==sha(archive)
    receipt=pd.DataFrame([dict(run_id=RUN_ID,key=key,generation=generation,progress=progress,sha256=sha(archive),bytes=archive.stat().st_size)])
    local=CACHE/f'{key}_receipt.csv';receipt.to_csv(local,index=False);shutil.copyfile(local,folder/f'slot{slot}.csv')
    reread=pd.read_csv(folder/f'slot{slot}.csv').iloc[0];assert reread.sha256==sha(archive) and int(reread.generation)==generation
    persistence_events.append(dict(key=key,event='saved',detail=f'generation={generation}, progress={progress}'))
def restore_snapshot(key):
    latest=read_snapshot(key)
    if latest is None:return False
    with zipfile.ZipFile(latest[1]) as z:
        inventory=pd.read_csv(io.BytesIO(z.read('snapshot_inventory.csv')))
        for r in inventory.itertuples():
            target=RUN_OUT.joinpath(*archive_relative(r.path).parts);assert target.resolve().is_relative_to(RUN_OUT.resolve())
            atomic_bytes(target,z.read(r.path))
    print('Đã phục hồi Drive:',key,flush=True);return True

## 3. Nguồn thu, độ dài và metadata sinh học
Đếm riêng clip/window và nguồn đã biết. Giá trị source trống là **chưa biết**, không được đếm thành một phiên thu độc lập. Domain D5 không tự được gán là microphone hoặc cảm biến quang học.

Metadata HumBug chỉ được liên kết bằng ID A đã ghi. Giới tính/thiết bị thiếu vẫn để thiếu; không suy ra giới tính từ một đỉnh phổ. Các bảng bằng chứng A được hiển thị lại để đối chiếu với kết quả B.

In [ ]:
verify_cell('coverage')
selected=clip_lookup.reset_index().copy()
coverage=selected.groupby(['species','split','domain'],sort=True).agg(clips=('clip_id','size'),windows=('full_windows','sum'),
    median_seconds=('processed_seconds','median'),median_rms_dbfs=('rms_dbfs','median'),
    known_source_names=('source_name_proxy',lambda s:s.replace('',np.nan).nunique()),
    clips_with_known_source=('source_name_proxy',lambda s:s.ne('').sum())).reset_index()
save_table(coverage,'source_coverage');display(coverage)
for name in ['split_design_review','sensor_modality_evidence','reference_source_proxy_warnings']:
    p=ROOT/'results/A/preparation/tables'/(name+'.csv')
    if p.is_file():display(save_table(pd.read_csv(p),name))
meta=pd.read_csv(ROOT/'data/metadata/humbugdb_zenodo_0_0_2.csv',keep_default_na=False).set_index('id')
metadata_rows=[]
for r in selected.itertuples():
    ids=[int(float(v)) for v in re.findall(r'\d+(?:\.\d+)?',str(r.humbug_ids))] if r.humbug_ids!='' else []
    if not ids:metadata_rows.append(dict(clip_id=r.clip_id,species=r.species,split=r.split,domain=r.domain,humbug_id='',gender='',mic_type='',device_type='',name='',evidence='unmapped'))
    for i in ids:
        assert i in meta.index
        q=meta.loc[i];metadata_rows.append(dict(clip_id=r.clip_id,species=r.species,split=r.split,domain=r.domain,humbug_id=i,
            gender=q.gender,mic_type=q.mic_type,device_type=q.device_type,name=q['name'],evidence='HumBug_ID_mapping'))
metadata_links=save_table(pd.DataFrame(metadata_rows),'metadata_links')
metadata_summary=metadata_links.assign(gender=lambda d:d.gender.replace('','unknown'),device_type=lambda d:d.device_type.replace('','unknown')).groupby(
    ['species','split','domain','gender','device_type'],dropna=False).agg(clips=('clip_id','nunique')).reset_index()
save_table(metadata_summary,'metadata_summary');display(metadata_summary)
fig,axes=plt.subplots(1,2,figsize=(13,4))
for ax,field,title in zip(axes,['clips','windows'],['Số clip theo loài và split','Số window theo loài và split']):
    data=coverage.groupby(['species','split'])[field].sum().unstack();data.plot.bar(ax=ax);ax.set_title(title);ax.set_ylabel(field);ax.tick_params(axis='x',rotation=20)
save_figure('01_coverage')

## 4. Sai ở đâu: window, clip và mức tự tin
Confusion matrix lấy từ dự đoán full đã kiểm ở mục 2. Hàng là nhãn thật, cột là dự đoán. Điểm softmax cao không bảo đảm dự đoán đúng, nhất là khi khác nguồn thu. Train recall được đọc từ report B, không đánh giá lại bằng audio train ở notebook này.

In [ ]:
verify_cell('error_visuals')
fig,axes=plt.subplots(2,4,figsize=(16,7))
short=['aegypti','albopictus','dirus','minimus']
for ax,model in zip(axes.ravel(),MODELS):
    c=predictions_b[model];matrix=confusion_matrix(c.label_id,c.prediction,labels=LABELS)
    sns.heatmap(matrix,annot=True,fmt='d',cbar=False,cmap='Blues',xticklabels=short,yticklabels=short,ax=ax)
    ax.set_title(model);ax.set_xlabel('Dự đoán');ax.set_ylabel('Nhãn thật');ax.tick_params(axis='x',rotation=30)
save_figure('02_full_confusion')
fig,axes=plt.subplots(1,2,figsize=(14,4))
alb_summary.set_index('model')[['train_clip_recall','validation_clip_recall','validation_window_recall']].plot.bar(ax=axes[0]);axes[0].set_ylim(0,1.1);axes[0].set_ylabel('Recall');axes[0].tick_params(axis='x',rotation=25)
alb_summary.set_index('model')[['mean_p_albopictus','mean_p_minimus']].plot.bar(ax=axes[1]);axes[1].set_ylabel('Điểm xác suất trung bình trên albopictus validation');axes[1].tick_params(axis='x',rotation=25)
save_figure('03_recall_confidence')
ds_errors=predictions_b['ds_cnn'].merge(selected[['clip_id','source_name_proxy','processed_seconds','rms_dbfs']],on='clip_id',validate='one_to_one')
ds_errors['correct']=ds_errors.prediction.eq(ds_errors.label_id)
save_table(ds_errors,'ds_cnn_clip_diagnostics')

## 5. Khảo sát audio: một window thật ở giữa mỗi clip
Chọn window trong manifest có tâm gần giữa clip nhất; giữ đúng sample/hop A. Mỗi clip đóng góp một window, giúp clip dài không lấn át khi so sánh nguồn. Đây khác cách lấy mọi window để train/score B.

Đo RMS, DC, clipping, tỷ lệ năng lượng theo dải, spectral centroid và độ tuần hoàn. PSD Welch bỏ DC theo từng frame. **Centroid/đỉnh phổ không được gọi là tần số đập cánh đã xác minh.** Lưu ID, vị trí sample và cả số window của clip để truy nguyên.

In [ ]:
verify_cell('acoustics')
center_rows=[]
for clip_id,frame in allowed_windows.groupby('clip_id',sort=True):
    length=int(clip_lookup.loc[clip_id,'processed_frames'])
    center_rows.append(frame.loc[((frame.start_sample+frame.end_sample)/2-length/2).abs().idxmin()])
center=pd.DataFrame(center_rows).sort_values(['species','split','domain','clip_id']).reset_index(drop=True)
if RUN_MODE=='smoke':
    picked=[]
    for _,frame in center.groupby(['species','split','domain'],sort=True):
        ids=np.random.default_rng(SEED).permutation(len(frame))[:SMOKE_CLIPS_PER_GROUP];picked.append(frame.iloc[ids])
    center=pd.concat(picked).sort_values('clip_id').reset_index(drop=True)
save_table(center.drop(columns=['path']),'acoustic_window_selection')
waveforms=[];spectra=[];mel_raw=[];acoustic_rows=[]
for i,r in enumerate(center.itertuples()):
    x=read_clip(r.clip_id)[int(r.start_sample):int(r.end_sample)].copy();assert x.shape==(2400,)
    f,p=signal.welch(x.astype('float64'),fs=8000,window=HANN,nperseg=256,noverlap=128,detrend='constant')
    q=p/max(p.sum(),1e-30);rms=float(np.sqrt(np.mean(x.astype('float64')**2)));z=x.astype('float64')-float(x.mean())
    ac=signal.correlate(z,z,mode='full',method='fft')[len(z)-1:];ac/=max(ac[0],1e-30)
    peaks,_=signal.find_peaks(ac,height=0);peaks=peaks[(peaks>=6)&(peaks<=53)]
    acoustic_rows.append(dict(clip_id=r.clip_id,window_id=r.window_id,species=r.species,label_id=r.label_id,split=r.split,domain=r.domain,
        rms_dbfs=20*np.log10(max(rms,1e-15)),peak=float(np.max(np.abs(x))),dc_mean=float(x.mean()),
        clipped_fraction=float((np.abs(x)>=.999).mean()),zero_fraction=float((x==0).mean()),
        centroid_hz=float((f*q).sum()),low_0_200=float(q[f<200].sum()),band_200_1000=float(q[(f>=200)&(f<1000)].sum()),
        high_1000_4000=float(q[f>=1000].sum()),periodicity_confidence=float(ac[peaks].max()) if len(peaks) else 0.))
    waveforms.append(x);spectra.append(10*np.log10(np.maximum(q,1e-12)));mel_raw.append(logmel(x))
    if (i+1)%200==0:print('Phân tích audio',i+1,'/',len(center),flush=True)
acoustics=save_table(pd.DataFrame(acoustic_rows),'acoustic_measurements')
waveforms=np.stack(waveforms);spectra=np.stack(spectra);mel_raw=np.stack(mel_raw)
check('Audio minh họa/đo đều là window A_v1',len(center)==len(acoustics) and np.isfinite(mel_raw).all())
features=['rms_dbfs','centroid_hz','low_0_200','high_1000_4000','periodicity_confidence']
summary=acoustics.groupby(['species','split','domain'])[features].agg(['count','median',lambda s:s.quantile(.25),lambda s:s.quantile(.75)])
summary.columns=['_'.join(map(str,c)).replace('<lambda_0>','q25').replace('<lambda_1>','q75') for c in summary.columns]
save_table(summary.reset_index(),'acoustic_summary');display(summary)
# Liên kết đặc trưng với lỗi của DS-CNN, không coi tương quan là nguyên nhân.
val_features=acoustics[acoustics.split.eq('validation')].merge(ds_errors[['clip_id','prediction','correct','p0','p1','p2','p3']],on='clip_id',validate='one_to_one')
save_table(val_features,'validation_acoustics_predictions')

## 6. So sánh phổ, năng lượng và các loài đối chứng
Phổ mỗi window được chia tổng năng lượng trước khi so sánh: khác biệt còn lại mô tả hình dạng phổ, không đơn thuần là âm lượng. Đường là median; vùng tô là phân vị 25–75% giữa các clip được khảo sát, không phải khoảng tin cậy độc lập theo phiên thu.

Các loài/nguồn đối chứng giúp tránh kết luận mọi khác biệt D1–D5 đều là đặc điểm sinh học của albopictus. Khi một nhãn chỉ có D5 ở train, không thể tách hoàn toàn ảnh hưởng loài và nguồn chỉ bằng biểu đồ.

In [ ]:
verify_cell('acoustic_visuals')
fig,axes=plt.subplots(2,2,figsize=(14,8))
for ax,feature in zip(axes.ravel(),['rms_dbfs','centroid_hz','low_0_200','high_1000_4000']):
    sns.boxplot(data=acoustics,x='species',y=feature,hue='split',showfliers=False,ax=ax);ax.tick_params(axis='x',rotation=15);ax.set_xlabel('');ax.set_title(feature)
save_figure('04_acoustic_distributions')
fig,axes=plt.subplots(1,2,figsize=(14,4))
for (split,domain),group in acoustics[acoustics.label_id.eq(1)].groupby(['split','domain']):
    q=np.percentile(spectra[group.index],[25,50,75],axis=0);axes[0].plot(FREQUENCIES,q[1],label=f'albopictus {split}/{domain}, n={len(group)}');axes[0].fill_between(FREQUENCIES,q[0],q[2],alpha=.15)
axes[0].set(title='Albopictus: PSD tương đối',xlabel='Hz',ylabel='dB, chuẩn hóa tổng năng lượng',xlim=(0,4000));axes[0].legend(fontsize=8)
for (label,split,domain),group in acoustics.groupby(['label_id','split','domain']):
    if (label==1) or (label==3 and split=='train' and domain in ('D1','D4')):
        axes[1].plot(FREQUENCIES,np.median(spectra[group.index],axis=0),label=f'{LABEL_NAMES[label]} {split}/{domain}')
axes[1].set(title='Có thêm minimus làm đối chứng',xlabel='Hz',ylabel='dB tương đối',xlim=(0,4000));axes[1].legend(fontsize=8)
save_figure('05_relative_spectra')

## 7. Chuẩn hóa B hiện tại có làm hai nguồn tương đương không?
Đọc mean/std **đã fit trên toàn bộ train B** từ checkpoint DS-CNN full; không fit lại bằng validation hoặc bằng vài mẫu đang vẽ. Hiển thị log-mel trước/sau chuẩn hóa với cùng thang màu giữa các nhóm.

Chuẩn hóa theo mel bin là phép biến đổi chung. Nó không tự xóa nhiễu, cân bằng mức âm từng clip, hoặc bù đáp ứng microphone. Biểu đồ ở đây kiểm tra trực tiếp điều đó; chưa huấn luyện cách chuẩn hóa mới.

In [ ]:
verify_cell('normalization')
payload=baseline_bytes('models/ds_cnn/best.pt')
best=torch.load(io.BytesIO(payload),map_location='cpu',weights_only=True)
assert best['run_id']==BASELINE_RUN_ID and best['label_names']==LABEL_NAMES and best['model_name']=='ds_cnn'
statistics=best['normalization'];mean=np.asarray(statistics['mel_mean'],dtype='float32');std=np.asarray(statistics['mel_std'],dtype='float32')
assert mean.shape==std.shape==(40,) and np.isfinite(mean).all() and (std>0).all()
assert int(statistics['fit_windows'])==len(allowed_windows[allowed_windows.split.eq('train')])
mel_normalized=(mel_raw-mean[None,:,None])/std[None,:,None]
normalization=pd.DataFrame({'mel_bin':np.arange(40),'train_mean':mean,'train_std':std});save_table(normalization,'baseline_normalization')
check('Chuẩn hóa đang minh họa lấy đúng thống kê full train B',int(statistics['fit_windows'])==41837)
fig,axes=plt.subplots(2,2,figsize=(12,7));groups=[]
for split in ['train','validation']:
    indices=acoustics.index[acoustics.label_id.eq(1)&acoustics.split.eq(split)].to_numpy();assert len(indices)
    groups.append(indices)
vmin,vmax=np.percentile(mel_raw[np.r_[*groups]],[2,98]);zmin,zmax=np.percentile(mel_normalized[np.r_[*groups]],[2,98])
for row,(split,indices) in enumerate(zip(['train','validation'],groups)):
    for col,(arr,lo,hi,title) in enumerate([(mel_raw,vmin,vmax,'Log-mel trước chuẩn hóa'),(mel_normalized,zmin,zmax,'Sau mean/std train của B')]):
        image=axes[row,col].imshow(np.median(arr[indices],axis=0),origin='lower',aspect='auto',vmin=lo,vmax=hi,cmap='magma');axes[row,col].set_title(f'Albopictus {split}: {title}');axes[row,col].set_xlabel('Frame STFT');axes[row,col].set_ylabel('Mel bin');fig.colorbar(image,ax=axes[row,col])
save_figure('06_before_after_B_normalization')

## 8. Nghe mẫu và xem waveform/spectrogram cùng thang màu
Mỗi nhóm albopictus train/D5, albopictus validation/D1 và minimus train/D1 lấy clip có centroid gần median nhóm; tie-break theo clip ID. Đây là ví dụ điển hình theo một chỉ số, không thay thế nghe nhiều mẫu. Bảng lưu chính xác quy tắc và ID; có thể dùng bảng acoustic để tự chọn thêm.

Trình phát dùng `normalize=False` để không âm thầm cân bằng âm lượng. Có thể tăng âm lượng thiết bị nghe. Khi nghe, ghi nhận tiếng cánh rõ/không rõ, nhiễu trội và các đoạn bất thường; không tự xác nhận giới tính hoặc nhãn chỉ bằng tai.

In [ ]:
verify_cell('examples')
example_indices=[]
for label,split,domain in [(1,'train','D5'),(1,'validation','D1'),(3,'train','D1')]:
    group=acoustics[acoustics.label_id.eq(label)&acoustics.split.eq(split)&acoustics.domain.eq(domain)].copy()
    group['distance']=(group.centroid_hz-group.centroid_hz.median()).abs()
    if len(group):example_indices.append(int(group.sort_values(['distance','clip_id']).index[0]))
examples=acoustics.loc[example_indices].copy();examples['selection_rule']='nearest_group_median_centroid_then_clip_id';examples['listening_status']='not_reviewed'
save_table(examples,'listening_examples');display(examples)
fig,axes=plt.subplots(len(example_indices),2,figsize=(12,3*len(example_indices)),squeeze=False)
vmin,vmax=np.percentile(mel_raw[example_indices],[2,98])
for row,i in enumerate(example_indices):
    r=acoustics.loc[i];axes[row,0].plot(np.arange(2400)/8000,waveforms[i]);axes[row,0].set(title=f'{r.clip_id}: {r.species}, {r.split}/{r.domain}',xlabel='Giây',ylabel='Biên độ')
    image=axes[row,1].imshow(mel_raw[i],origin='lower',aspect='auto',cmap='magma',vmin=vmin,vmax=vmax);axes[row,1].set(xlabel='Frame',ylabel='Mel bin');fig.colorbar(image,ax=axes[row,1])
    display(Markdown(f'**{r.clip_id} — {r.split}/{r.domain}**'));display(Audio(waveforms[i],rate=8000,normalize=False))
save_figure('07_listening_examples')

## 9. Kết luận có điều kiện và bàn giao sang 06
Notebook tự tổng hợp các đo lường, không biến chúng thành kết luận nhân quả. Khi D5 chỉ gắn với albopictus ở train, dữ liệu hiện tại không cho phép tách hoàn toàn ảnh hưởng nguồn và sinh học. Nguồn D1 cùng tên không đại diện nhiều phiên thu độc lập. Nhãn/giới tính/nguồn D5 còn thiếu cần được xác minh bên ngoài.

06 kiểm tra từng thay đổi với DS-CNN: RMS, high-pass, class weight, balanced sampling. Cải thiện của high-pass chỉ hỗ trợ giả thuyết dải thấp có ảnh hưởng; không chứng minh nhiễu là nguyên nhân duy nhất. Nếu không cải thiện, chưa thể kết luận model là nguyên nhân duy nhất hoặc dữ liệu không dùng được.

In [ ]:
def write_report(title,narrative,table_names):
    sections=[f'<h1>{html.escape(title)}</h1>',f'<p>Mode: <b>{RUN_MODE}</b> | Run: {RUN_ID[:16]} | {datetime.now().isoformat(timespec="seconds")}</p>',narrative]
    for name in table_names:
        p=TABLES/(name+'.csv')
        if p.exists():
            sections.extend([f'<h2>{html.escape(name)}</h2>','<div class="scroll">'+pd.read_csv(p).to_html(index=False,border=0,float_format=lambda x:f'{x:.4f}')+'</div>'])
    for p in sorted(FIGURES.glob('*.png')):sections.append(f'<h2>{html.escape(p.stem)}</h2><img src="figures/{p.name}" alt="{html.escape(p.stem)}">')
    text='<!doctype html><html lang="vi"><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1"><title>'+html.escape(title)+'</title><style>body{font:16px/1.55 system-ui;max-width:1200px;margin:32px auto;padding:0 20px;color:#172a3a}table{border-collapse:collapse;font-size:13px}td,th{padding:7px;border-bottom:1px solid #ddd;text-align:left}th{background:#edf3f7}.scroll{overflow:auto}img{max-width:100%;height:auto}h2{margin-top:32px}p{max-width:1000px}</style>'+''.join(sections)+'</html>'
    atomic_bytes(RUN_OUT/'report.html',text.encode('utf-8'))
    print('Report:',RUN_OUT/'report.html')

verify_cell("report_helpers")

In [ ]:
verify_cell('report')
rows=[]
alb_audio=acoustics[acoustics.label_id.eq(1)]
for split,g in alb_audio.groupby('split'):
    rows.append(dict(evidence='measured_audio',item='albopictus_'+split,details=f'n={len(g)}, median low<200Hz={g.low_0_200.median():.3f}, centroid={g.centroid_hz.median():.1f}Hz'))
rows += [dict(evidence='verified_full_predictions',item='validation_failure',details=f'{int((alb_summary.validation_clip_recall==0).sum())}/{len(alb_summary)} model có albopictus clip recall=0'),
         dict(evidence='hypothesis',item='source_confounding',details='Train albopictus chỉ D5; validation D1. Cần đối chứng 06; chưa xác định tác nhân nhân quả.'),
         dict(evidence='unknown',item='biology_sensor',details='Giới tính và cảm biến/phiên thu D5 chưa đủ mapping; không tự suy ra từ phổ.'),
         dict(evidence='next_experiment',item='06',details='Baseline đối chứng cùng kiến trúc, split và seed; từng lần đổi một yếu tố.')]
save_table(pd.DataFrame(rows),'findings_and_limits')
check('Không đọc audio test',not accessed&test_ids and accessed<=allowed_ids)
check('Có đo thực audio train/validation',len(accessed)>0)
for p,before in PROTECTED.items():check('Không sửa '+str(p.relative_to(ROOT)),sha(p)==before)
check('Các cell đã chạy khớp source trên kernel',set(core_cells)<=verified_tags)
save_table(pd.DataFrame(checks),'validation_checks');save_table(pd.DataFrame([dict(clip_id=i,split=clip_lookup.loc[i,'split']) for i in sorted(accessed)]),'audio_access_log')
write_report('B05 — Điều tra lỗi albopictus',
    '<p>Full baseline được kiểm lại ở hai mức window/clip. Khảo sát audio dùng một window giữa mỗi clip; phổ được chuẩn hóa tổng năng lượng. '+
    ('SMOKE: chỉ ít mẫu audio, không dùng kết luận phân bố. ' if RUN_MODE=='smoke' else '')+
    'Khác biệt nguồn thu là giả thuyết có bằng chứng hỗ trợ; chưa xác định nhiễu, thiết bị, giới tính hay nhãn là tác nhân. Không đọc audio test.</p>',
    ['findings_and_limits','albopictus_failure_summary','baseline_metric_audit','prediction_agreement','source_coverage','metadata_summary','acoustic_summary','listening_examples','validation_checks'])
files=[RUN_OUT/'report.html',RUN_OUT/'source_notebook.ipynb',*TABLES.glob('*.csv'),*FIGURES.glob('*.png')]
save_snapshot('report',files,1)
print('Hoàn tất 05. Dùng findings_and_limits và audio để lựa chọn/diễn giải thử nghiệm 06.')